In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *

In [0]:
ls=dbutils.fs.ls('abfss://synapse@testexpstorageacc.dfs.core.windows.net/testdata/')

In [0]:
lstgt=[]
for i in ls:
    lstgt.append(i.path+','+i.name+','+str(i.size)+','+str(i.modificationTime))



In [0]:
testdata=spark.read.format('csv')\
.option('header','true')\
.load("abfss://synapse@testexpstorageacc.dfs.core.windows.net/testdata/CreditCard.csv")


In [0]:
salesorderdetail=spark.table('test_exp_bricks.default.salesorderdetail')\
    .withColumns({'testcol': lit(1).cast("int"), 'rno':row_number().over(Window.partitionBy("SalesOrderID").orderBy(col("LineTotal").desc()))})

In [0]:
salesorderdetail.columns

In [0]:
df=salesorderdetail\
    .select("SalesOrderID","UnitPrice",sum("UnitPrice").over(Window.partitionBy("SalesOrderID").orderBy(col("SalesOrderID").asc())).alias("runningtotal"))

display(df)

In [0]:
df=salesorderdetail\
    .select("SalesOrderID","UnitPrice","ProductID")\
        .groupBy("SalesOrderID","ProductID").agg(sum("UnitPrice").alias("TotalUnitPrice"))

display(df)

In [0]:
df=salesorderdetail.repartition(10)

In [0]:
print(df.select(spark_partition_id()))

In [0]:
%sql
CREATE TABLE test_exp_bricks.default.salesorderdetail_test AS SELECT * FROM test_exp_bricks.default.salesorderdetail WHERE 1=2

In [0]:
salesorderdetail.write.mode("overwrite").option("mergeSchema","true").saveAsTable('test_exp_bricks.default.salesorderdetail_test')

In [0]:
%sql
select * from test_exp_bricks.default.salesorderdetail_test